In [1]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Pauli
from itertools import product


def generate_target_paulis(target_weight, num_qubits):
    """Generates Pauli strings for the 7 data qubits of a specific weight."""
    paulis = []
    bases = ['I', 'X', 'Y', 'Z']
    for comb in product(bases, repeat=7):
        w = sum(1 for c in comb if c != 'I')
        if w == target_weight:
            # Qiskit orders strings qN...q0. Data is q0-q6.
            q_chars = ['I'] * num_qubits
            labels = []
            for i, c in enumerate(comb):
                if c != 'I':
                    q_chars[(num_qubits - 1) - i] = c
                    labels.append(f"{c}{i}")

            qiskit_str = "".join(q_chars)
            paulis.append((qiskit_str, " ".join(labels)))
    return paulis

def project_and_normalize(sv_data, num_qubits):
    """Projects onto the flags=0 subspace dynamically and normalizes."""
    data = sv_data.copy()

    # Create a dynamic bitmask for all flag qubits (q7 to num_qubits-1)
    flag_mask = sum(1 << i for i in range(7, num_qubits))
    sv_size = 2**num_qubits

    for idx in range(sv_size):
        if (idx & flag_mask) != 0:
            data[idx] = 0.0

    prob = np.sum(np.abs(data)**2)
    if prob > 1e-8:
        data = data / np.sqrt(prob)
    return data, prob

def analyze_fault_tolerance(base_qc):
    num_qubits = base_qc.num_qubits
    num_instructions = len(base_qc.data)

    # Generate ideal accepted state
    ideal_sv = Statevector(base_qc)
    ideal_acc_data, ideal_prob = project_and_normalize(ideal_sv.data, num_qubits)

    w1_paulis = generate_target_paulis(1, num_qubits)
    w2_paulis = generate_target_paulis(2, num_qubits)

    paulis_1q = ['X', 'Y', 'Z']
    paulis_2q = [''.join(p) for p in product(['I', 'X', 'Y', 'Z'], repeat=2) if ''.join(p) != 'II']

    print(f"Running Absolute Fidelity Verification ({num_qubits} Qubits)...")
    print("Failure Criteria: Faulty state cannot be corrected by ANY weight <= 1 operator.\n")

    failures_count = 0

    for loc in range(num_instructions):
        instr = base_qc.data[loc]
        q_indices = [base_qc.find_bit(q).index for q in instr.qubits]
        errors_to_test = paulis_2q if len(q_indices) == 2 else paulis_1q

        for error_str in errors_to_test:
            test_qc = QuantumCircuit(num_qubits)

            # 1. Run up to fault location
            for i in range(loc + 1):
                test_qc.append(base_qc.data[i].operation, base_qc.data[i].qubits)

            # 2. Inject fault
            for char_idx, q_idx in enumerate(q_indices):
                p_err = error_str[char_idx]
                if p_err == 'X': test_qc.x(q_idx)
                elif p_err == 'Y': test_qc.y(q_idx)
                elif p_err == 'Z': test_qc.z(q_idx)

            # 3. Complete circuit
            for i in range(loc + 1, num_instructions):
                test_qc.append(base_qc.data[i].operation, base_qc.data[i].qubits)

            sv = Statevector(test_qc)
            err_acc_data, err_prob = project_and_normalize(sv.data, num_qubits)

            # If the flags caught it perfectly, move on
            if err_prob < 1e-6:
                continue

            # --- EVALUATE TRUE PHYSICAL DISTANCE ---

            # Check 1: Is it physically equivalent to NO error? (Weight 0)
            f0 = np.abs(np.vdot(err_acc_data, ideal_acc_data))**2
            if f0 > 0.9999:
                continue

            # Check 2: Is it physically equivalent to ANY Weight 1 error?
            is_w1 = False
            for p_str, label in w1_paulis:
                p_ideal_sv = Statevector(ideal_acc_data).evolve(Pauli(p_str))
                f1 = np.abs(np.vdot(err_acc_data, p_ideal_sv.data))**2
                if f1 > 0.9999:
                    is_w1 = True
                    break

            if is_w1:
                continue

            # Definitively a code-breaking failure
            failures_count += 1
            matched_w2 = False

            for p_str, label in w2_paulis:
                p_ideal_sv = Statevector(ideal_acc_data).evolve(Pauli(p_str))
                f2 = np.abs(np.vdot(err_acc_data, p_ideal_sv.data))**2
                if f2 > 0.9999:
                    print(f"❌ TRUE FAULT-TOLERANCE FAILURE")
                    print(f"  - Injection: Gate {loc} ({instr.operation.name}) on {q_indices}")
                    print(f"  - Fault Type: {error_str}")
                    print(f"  - Escaped Data Error: {label} (Exact Weight 2 Match)")
                    print(f"  - Acceptance Prob: {err_prob:.4f}")
                    print("-" * 60)
                    matched_w2 = True
                    break

            if not matched_w2:
                print(f"❌ TRUE FAULT-TOLERANCE FAILURE")
                print(f"  - Injection: Gate {loc} ({instr.operation.name}) on {q_indices}")
                print(f"  - Fault Type: {error_str}")
                print(f"  - Escaped Data Error: Coherent/Complex State (Uncorrectable by d=3)")
                print(f"  - Acceptance Prob: {err_prob:.4f}")
                print("-" * 60)

    if failures_count == 0:
        print("✅ Success! All accepted physical states are correctable via distance-3 recovery.")
    else:
        print(f"Analysis complete. Found {failures_count} true uncorrectable physical violations.")

In [2]:
def css_pft_arbitrary_angle():
    qc = QuantumCircuit(10)
    # Block 1
    qc.h([2, 3, 4, 5])
    qc.cx(2, 7)
    qc.cx(3, 1)
    qc.cx(5, 6)
    qc.cx(2, 0)
    qc.rzz(0.1 * 2 * np.pi, 2, 3)
    qc.cx(2, 7)
    qc.cx(4, 3)
    qc.cx(5, 2)
    qc.cx(4, 6)
    qc.cx(2, 1)
    qc.cx(3, 5)

    # Blocks 2-5
    qc.cx(0, 8); qc.h(9); qc.cx(9, 1)
    qc.cx(1, 8); qc.cx(9, 0)
    qc.cx(4, 8); qc.cx(9, 5)
    qc.cx(5, 8); qc.cx(9, 4); qc.h(9)
    return qc


circ = css_pft_arbitrary_angle()
analyze_fault_tolerance(circ)

circ.draw(fold=-1)

Running Absolute Fidelity Verification (10 Qubits)...
Failure Criteria: Faulty state cannot be corrected by ANY weight <= 1 operator.

❌ TRUE FAULT-TOLERANCE FAILURE
  - Injection: Gate 8 (rzz) on [2, 3]
  - Fault Type: ZZ
  - Escaped Data Error: Coherent/Complex State (Uncorrectable by d=3)
  - Acceptance Prob: 1.0000
------------------------------------------------------------
❌ TRUE FAULT-TOLERANCE FAILURE
  - Injection: Gate 11 (cx) on [5, 2]
  - Fault Type: XY
  - Escaped Data Error: Z2 X6 (Exact Weight 2 Match)
  - Acceptance Prob: 1.0000
------------------------------------------------------------
❌ TRUE FAULT-TOLERANCE FAILURE
  - Injection: Gate 20 (cx) on [4, 8]
  - Fault Type: XZ
  - Escaped Data Error: X4 Z5 (Exact Weight 2 Match)
  - Acceptance Prob: 1.0000
------------------------------------------------------------
❌ TRUE FAULT-TOLERANCE FAILURE
  - Injection: Gate 21 (cx) on [9, 5]
  - Fault Type: XZ
  - Escaped Data Error: X4 Z5 (Exact Weight 2 Match)
  - Acceptance Pr

┌───┐                                                  ┌───┐                         
q_0: ───────────────┤ X ├────────────■─────────────────────────────────────┤ X ├─────────────────────────
               ┌───┐└─┬─┘            │                 ┌───┐     ┌───┐     └─┬─┘                         
q_1: ──────────┤ X ├──┼──────────────┼─────────────────┤ X ├─────┤ X ├──■────┼───────────────────────────
     ┌───┐     └─┬─┘  │              │            ┌───┐└─┬─┘     └─┬─┘  │    │                           
q_2: ┤ H ├──■────┼────■───■──────────┼────■───────┤ X ├──■─────────┼────┼────┼───────────────────────────
     ├───┤  │    │        │ZZ(π/5)   │    │  ┌───┐└─┬─┘            │    │    │                           
q_3: ┤ H ├──┼────■────────■──────────┼────┼──┤ X ├──┼─────────■────┼────┼────┼───────────────────────────
     ├───┤  │                        │    │  └─┬─┘  │         │    │    │    │                 ┌───┐     
q_4: ┤ H ├──┼────────────────────────┼────┼────■────┼────■────┼────┼────┼────┼────■────────────┤ X ├─────
     ├───┤  │                        │    │         │    │  ┌─┴─┐  │    │    │    │  ┌───┐     └─┬─┘     
q_5: ┤ H ├──┼────■───────────────────┼────┼─────────■────┼──┤ X ├──┼────┼────┼────┼──┤ X ├──■────┼───────
     └───┘  │  ┌─┴─┐                 │    │            ┌─┴─┐└───┘  │    │    │    │  └─┬─┘  │    │       
q_6: ───────┼──┤ X ├─────────────────┼────┼────────────┤ X ├───────┼────┼────┼────┼────┼────┼────┼───────
          ┌─┴─┐└───┘                 │  ┌─┴─┐          └───┘       │    │    │    │    │    │    │       
q_7: ─────┤ X ├──────────────────────┼──┤ X ├──────────────────────┼────┼────┼────┼────┼────┼────┼───────
          └───┘                    ┌─┴─┐└───┘                      │  ┌─┴─┐  │  ┌─┴─┐  │  ┌─┴─┐  │       
q_8: ──────────────────────────────┤ X ├───────────────────────────┼──┤ X ├──┼──┤ X ├──┼──┤ X ├──┼───────
     ┌───┐                         └───┘                           │  └───┘  │  └───┘  │  └───┘  │  ┌───┐
q_9: ┤ H ├─────────────────────────────────────────────────────────■─────────■─────────■─────────■──┤ H ├
     └───┘                                                                                          └───┘

In [3]:
def true_pft_arbitrary_angle():
    qc = QuantumCircuit(11) # Updated to 11 qubits
    # Block 1
    qc.h([2, 3, 4, 5, 9, 10])
    qc.cx(2, 7)
    qc.cx(3, 1)
    qc.cz(10, 9)
    qc.cx(2, 0)
    qc.rzz(0.25 * np.pi, 2, 3)
    qc.cx(2, 7)

    qc.cx(4, 3)
    qc.cx(10, 8)

    qc.cx(5, 2)
    qc.cx(5, 6)

    qc.cx(4, 6)
    qc.cx(2, 1)
    qc.cx(3, 5)

    # Blocks 2-5 (Updated with q10 and CZs)
    qc.cx(0, 8); qc.cx(9, 1)
    qc.cx(1, 8); qc.cx(9, 0)
    qc.cx(4, 8); qc.cx(9, 5)
    qc.cz(10, 9)
    qc.cx(10, 8)
    qc.cx(5, 8); qc.cx(9, 4)
    qc.h([9, 10])
    return qc


circ = true_pft_arbitrary_angle()
analyze_fault_tolerance(circ)

circ.draw(fold=-1)

Running Absolute Fidelity Verification (11 Qubits)...
Failure Criteria: Faulty state cannot be corrected by ANY weight <= 1 operator.

❌ TRUE FAULT-TOLERANCE FAILURE
  - Injection: Gate 10 (rzz) on [2, 3]
  - Fault Type: ZZ
  - Escaped Data Error: Coherent/Complex State (Uncorrectable by d=3)
  - Acceptance Prob: 1.0000
------------------------------------------------------------
Analysis complete. Found 1 true uncorrectable physical violations.


┌───┐                                                       ┌───┐                            
 q_0: ───────────────┤ X ├────────────■──────────────────────────────────────────┤ X ├────────────────────────────
                ┌───┐└─┬─┘            │                 ┌───┐          ┌───┐     └─┬─┘                            
 q_1: ──────────┤ X ├──┼──────────────┼─────────────────┤ X ├──────────┤ X ├──■────┼──────────────────────────────
      ┌───┐     └─┬─┘  │              │            ┌───┐└─┬─┘          └─┬─┘  │    │                              
 q_2: ┤ H ├──■────┼────■───■──────────┼────■───────┤ X ├──■──────────────┼────┼────┼──────────────────────────────
      ├───┤  │    │        │ZZ(π/4)   │    │  ┌───┐└─┬─┘                 │    │    │                              
 q_3: ┤ H ├──┼────■────────■──────────┼────┼──┤ X ├──┼──────────────■────┼────┼────┼──────────────────────────────
      ├───┤  │                        │    │  └─┬─┘  │              │    │    │    │                    ┌───┐     
 q_4: ┤ H ├──┼────────────────────────┼────┼────■────┼─────────■────┼────┼────┼────┼────■───────────────┤ X ├─────
      ├───┤  │                        │    │         │         │  ┌─┴─┐  │    │    │    │  ┌───┐        └─┬─┘     
 q_5: ┤ H ├──┼────────────────────────┼────┼─────────■────■────┼──┤ X ├──┼────┼────┼────┼──┤ X ├──────────┼────■──
      └───┘  │                        │    │            ┌─┴─┐┌─┴─┐└───┘  │    │    │    │  └─┬─┘          │    │  
 q_6: ───────┼────────────────────────┼────┼────────────┤ X ├┤ X ├───────┼────┼────┼────┼────┼────────────┼────┼──
           ┌─┴─┐                      │  ┌─┴─┐          └───┘└───┘       │    │    │    │    │            │    │  
 q_7: ─────┤ X ├──────────────────────┼──┤ X ├───────────────────────────┼────┼────┼────┼────┼────────────┼────┼──
           └───┘┌───┐               ┌─┴─┐└───┘                           │  ┌─┴─┐  │  ┌─┴─┐  │     ┌───┐  │  ┌─┴─┐
 q_8: ──────────┤ X ├───────────────┤ X ├────────────────────────────────┼──┤ X ├──┼──┤ X ├──┼─────┤ X ├──┼──┤ X ├
      ┌───┐     └─┬─┘               └───┘                                │  └───┘  │  └───┘  │     └─┬─┘  │  ├───┤
 q_9: ┤ H ├──■────┼──────────────────────────────────────────────────────■─────────■─────────■───■───┼────■──┤ H ├
      ├───┤  │    │                                                                              │   │  ┌───┐└───┘
q_10: ┤ H ├──■────■──────────────────────────────────────────────────────────────────────────────■───■──┤ H ├─────
      └───┘                                                                                             └───┘

In [4]:
def true_ft_zero_state():
    qc = QuantumCircuit(10)
    qc.h([0, 4, 6])
    qc.cx(0, 1)
    qc.cx(4, 5)
    qc.cx(6, 3)

    qc.h([7, 9])
    qc.cz(7, 9)

    qc.cx(6, 5)
    qc.cx(0, 3)

    qc.cz(1, 7)

    qc.cx(4, 1)
    qc.cx(4, 2)
    qc.cx(3, 2)


    qc.cz(0, 7)
    # qc.cz(1, 7)
    # qc.cz(4, 7)

    qc.cx(6, 8)
    qc.cx(1, 8)
    qc.cx(2, 8)

    qc.cz(5, 9)
    qc.cz(1, 9)
    qc.cz(3, 9)

    qc.cz(7, 9)
    qc.h([7, 9])

    return qc


circ = true_ft_zero_state()
analyze_fault_tolerance(circ)

circ.draw(fold=-1)

Running Absolute Fidelity Verification (10 Qubits)...
Failure Criteria: Faulty state cannot be corrected by ANY weight <= 1 operator.

✅ Success! All accepted physical states are correctable via distance-3 recovery.


┌───┐                                                            
q_0: ┤ H ├──■─────────■───────────■───────────────────────────────────
     └───┘┌─┴─┐       │     ┌───┐ │                                   
q_1: ─────┤ X ├───────┼───■─┤ X ├─┼────────■────────■─────────────────
          └───┘       │   │ └─┬─┘ │ ┌───┐  │  ┌───┐ │                 
q_2: ─────────────────┼───┼───┼───┼─┤ X ├──┼──┤ X ├─┼───■─────────────
               ┌───┐┌─┴─┐ │   │   │ └─┬─┘  │  └─┬─┘ │   │             
q_3: ──────────┤ X ├┤ X ├─┼───┼───┼───┼────┼────■───┼───┼───■─────────
     ┌───┐     └─┬─┘└───┘ │   │   │   │    │        │   │   │         
q_4: ┤ H ├──■────┼────────┼───■───┼───■────┼────────┼───┼───┼─────────
     └───┘┌─┴─┐  │  ┌───┐ │       │        │        │   │   │         
q_5: ─────┤ X ├──┼──┤ X ├─┼───────┼───■────┼────────┼───┼───┼─────────
     ┌───┐└───┘  │  └─┬─┘ │       │   │    │        │   │   │         
q_6: ┤ H ├───────■────■───┼───■───┼───┼────┼────────┼───┼───┼─────────
     ├───┤                │   │   │   │    │        │   │   │    ┌───┐
q_7: ┤ H ├──■─────────────■───┼───■───┼────┼────────┼───┼───┼──■─┤ H ├
     └───┘  │               ┌─┴─┐     │  ┌─┴─┐      │ ┌─┴─┐ │  │ └───┘
q_8: ───────┼───────────────┤ X ├─────┼──┤ X ├──────┼─┤ X ├─┼──┼──────
     ┌───┐  │               └───┘     │  └───┘      │ └───┘ │  │ ┌───┐
q_9: ┤ H ├──■─────────────────────────■─────────────■───────■──■─┤ H ├
     └───┘                                                       └───┘

In [5]:
def true_ft_plus_state():
    qc = QuantumCircuit(10)
    qc.h([1, 2, 3, 5, 7, 8, 9])
    qc.cx(1, 0)
    qc.cx(5, 4)
    qc.cx(3, 6)

    qc.cz(9, 7)

    qc.cx(5, 6)
    qc.cx(3, 0)

    qc.cx(7, 1)

    qc.cx(1, 4)
    qc.cx(2, 4)
    qc.cx(2, 3)


    qc.cx(7, 0)
    # qc.cx(7, 1)
    # qc.cx(7, 4)

    qc.cx(8, 6)
    qc.cx(8, 1)
    qc.cx(8, 2)

    qc.cx(9, 5)
    qc.cx(9, 1)
    qc.cx(9, 3)

    qc.cz(9, 7)
    qc.h([7, 8, 9])

    return qc


circ = true_ft_plus_state()
analyze_fault_tolerance(circ)

circ.draw(fold=-1)

Running Absolute Fidelity Verification (10 Qubits)...
Failure Criteria: Faulty state cannot be corrected by ANY weight <= 1 operator.

✅ Success! All accepted physical states are correctable via distance-3 recovery.


┌───┐     ┌───┐          ┌───┐                                      
q_0: ─────┤ X ├─────┤ X ├──────────┤ X ├──────────────────────────────────────
     ┌───┐└─┬─┘     └─┬─┘┌───┐     └─┬─┘     ┌───┐     ┌───┐                  
q_1: ┤ H ├──■─────────┼──┤ X ├──■────┼───────┤ X ├─────┤ X ├──────────────────
     ├───┤            │  └─┬─┘  │    │       └─┬─┘     └─┬─┘┌───┐             
q_2: ┤ H ├────────────┼────┼────┼────┼────■────┼────■────┼──┤ X ├─────────────
     ├───┤            │    │    │    │    │    │  ┌─┴─┐  │  └─┬─┘┌───┐        
q_3: ┤ H ├───────■────■────┼────┼────┼────┼────┼──┤ X ├──┼────┼──┤ X ├────────
     └───┘┌───┐  │         │  ┌─┴─┐  │  ┌─┴─┐  │  └───┘  │    │  └─┬─┘        
q_4: ─────┤ X ├──┼─────────┼──┤ X ├──┼──┤ X ├──┼─────────┼────┼────┼──────────
     ┌───┐└─┬─┘  │         │  └───┘  │  ├───┤  │         │    │    │          
q_5: ┤ H ├──■────┼────■────┼─────────┼──┤ X ├──┼─────────┼────┼────┼──────────
     └───┘     ┌─┴─┐┌─┴─┐  │  ┌───┐  │  └─┬─┘  │         │    │    │          
q_6: ──────────┤ X ├┤ X ├──┼──┤ X ├──┼────┼────┼─────────┼────┼────┼──────────
     ┌───┐     └───┘└───┘  │  └─┬─┘  │    │    │         │    │    │     ┌───┐
q_7: ┤ H ├──■──────────────■────┼────■────┼────┼─────────┼────┼────┼───■─┤ H ├
     ├───┤  │                   │         │    │         │    │    │   │ ├───┤
q_8: ┤ H ├──┼───────────────────■─────────┼────■─────────┼────■────┼───┼─┤ H ├
     ├───┤  │                             │              │         │   │ ├───┤
q_9: ┤ H ├──■─────────────────────────────■──────────────■─────────■───■─┤ H ├
     └───┘                                                               └───┘

In [8]:
def true_ft_plus_i_state():
    qc = QuantumCircuit(10)
    qc.h([1, 2, 3, 5, 7, 8, 9])
    qc.cx(1, 0)
    qc.cx(5, 4)
    qc.cx(3, 6)

    qc.cz(9, 7)

    qc.cx(5, 6)
    qc.cx(3, 0)

    qc.cx(7, 1)

    qc.cx(1, 4)
    qc.cx(2, 4)
    qc.cx(2, 3)


    qc.cx(7, 0)
    # qc.cx(7, 1)
    # qc.cx(7, 4)

    qc.cx(8, 6)
    qc.cx(8, 1)
    qc.cx(8, 2)

    qc.cx(9, 5)
    qc.cx(9, 1)
    qc.cx(9, 3)

    qc.cz(9, 7)
    qc.h([7, 8, 9])

    qc.sdg([0, 1, 2, 3, 4, 5, 6])

    return qc


circ = true_ft_plus_i_state()
analyze_fault_tolerance(circ)

circ.draw(fold=-1)

Running Absolute Fidelity Verification (10 Qubits)...
Failure Criteria: Faulty state cannot be corrected by ANY weight <= 1 operator.

✅ Success! All accepted physical states are correctable via distance-3 recovery.


┌───┐     ┌───┐          ┌───┐┌─────┐                                           
q_0: ─────┤ X ├─────┤ X ├──────────┤ X ├┤ Sdg ├───────────────────────────────────────────
     ┌───┐└─┬─┘     └─┬─┘┌───┐     └─┬─┘└─────┘┌───┐       ┌───┐┌─────┐                   
q_1: ┤ H ├──■─────────┼──┤ X ├──■────┼─────────┤ X ├───────┤ X ├┤ Sdg ├───────────────────
     ├───┤            │  └─┬─┘  │    │         └─┬─┘       └─┬─┘└┬───┬┘┌─────┐            
q_2: ┤ H ├────────────┼────┼────┼────┼─────■─────┼─────■─────┼───┤ X ├─┤ Sdg ├────────────
     ├───┤            │    │    │    │     │     │   ┌─┴─┐   │   └─┬─┘ └┬───┬┘┌─────┐     
q_3: ┤ H ├───────■────■────┼────┼────┼─────┼─────┼───┤ X ├───┼─────┼────┤ X ├─┤ Sdg ├─────
     └───┘┌───┐  │         │  ┌─┴─┐  │   ┌─┴─┐   │  ┌┴───┴┐  │     │    └─┬─┘ └─────┘     
q_4: ─────┤ X ├──┼─────────┼──┤ X ├──┼───┤ X ├───┼──┤ Sdg ├──┼─────┼──────┼───────────────
     ┌───┐└─┬─┘  │         │  └───┘  │   ├───┤   │  ├─────┤  │     │      │               
q_5: ┤ H ├──■────┼────■────┼─────────┼───┤ X ├───┼──┤ Sdg ├──┼─────┼──────┼───────────────
     └───┘     ┌─┴─┐┌─┴─┐  │  ┌───┐  │   └─┬─┘   │  ├─────┤  │     │      │               
q_6: ──────────┤ X ├┤ X ├──┼──┤ X ├──┼─────┼─────┼──┤ Sdg ├──┼─────┼──────┼───────────────
     ┌───┐     └───┘└───┘  │  └─┬─┘  │     │     │  └─────┘  │     │      │          ┌───┐
q_7: ┤ H ├──■──────────────■────┼────■─────┼─────┼───────────┼─────┼──────┼──────■───┤ H ├
     ├───┤  │                   │          │     │           │     │      │      │   ├───┤
q_8: ┤ H ├──┼───────────────────■──────────┼─────■───────────┼─────■──────┼──────┼───┤ H ├
     ├───┤  │                              │                 │            │      │   ├───┤
q_9: ┤ H ├──■──────────────────────────────■─────────────────■────────────■──────■───┤ H ├
     └───┘                                                                           └───┘